# 03 · Write your own method

A method is a `Suggester` with two methods:

- `fit(train)`: called once per cross-validation fold, with the training scenarios only.
- `suggest(context, catalog, history=None, k=10)`: returns up to `k` facet ids, best first.

Below is a toy method: **environment-conditioned popularity**. It suggests the facets placed most often in the same environment in the training data, then falls back to overall popularity. The cell after it compares it against every paper baseline under identical conditions. LLM baselines replay `cache/llm/`, so no API key is needed.

In [ ]:
from collections import Counter, defaultdict
from contextxr.methods import Suggester


class EnvironmentPopularity(Suggester):
    name = "env_popularity (ours)"

    def fit(self, train):
        self.by_env, self.overall = defaultdict(Counter), Counter()
        for scenario in train:
            env = scenario.context.environment.lower()
            for widget in scenario.widgets:
                if widget.facet_id:
                    self.by_env[env][widget.facet_id] += 1
                    self.overall[widget.facet_id] += 1

    def suggest(self, context, catalog, history=None, k=10):
        local = self.by_env.get(context.environment.lower(), Counter())
        ranked = sorted(catalog.facets, key=lambda f: (-local[f], -self.overall[f], f))
        return [f for f in ranked if f not in (history or [])][:k]

In [ ]:
from IPython.display import Markdown
from contextxr import evaluate, load_minexrpp
from contextxr.methods.registry import build_method, paper_methods
from contextxr.runner import format_table

ds = load_minexrpp()
results = [evaluate(EnvironmentPopularity(), "initial", ds)]
for name in paper_methods():
    method, protocol = build_method(name, task="initial", dataset=ds)
    results.append(evaluate(method, "initial", ds, **protocol))

Markdown(format_table(results, sort_by="reduction"))

| method | n | manual | cost | reduction | Recall@10 | F1@10 | Hit@10 |
|---|---:|---:|---:|---:|---:|---:|---:|
| few_shot_llm[user+task] | 2 | 22.00 | 15.25 | -30.7% | 0.464 | 0.399 | 1.000 |
| few_shot_llm[user+env] | 94 | 20.07 | 15.53 | -22.7% | 0.319 | 0.229 | 0.862 |
| few_shot_llm[user] | 106 | 21.09 | 16.33 | -22.6% | 0.338 | 0.255 | 0.830 |
| few_shot_llm[environment] | 108 | 21.03 | 17.07 | -18.8% | 0.287 | 0.206 | 0.870 |
| zero_shot_llm | 109 | 21.20 | 17.66 | -16.7% | 0.228 | 0.168 | 0.761 |
| few_shot_llm[task] | 40 | 21.90 | 18.41 | -15.9% | 0.274 | 0.218 | 0.825 |
| relational | 109 | 21.20 | 18.39 | -13.3% | 0.259 | 0.193 | 0.798 |
| few_shot_llm[task+env] | 47 | 22.36 | 19.54 | -12.6% | 0.222 | 0.178 | 0.702 |
| env_popularity (ours) | 109 | 21.20 | 18.99 | -10.5% | 0.218 | 0.155 | 0.771 |
| popularity | 109 | 21.20 | 19.85 | -6.4% | 0.167 | 0.132 | 0.688 |

**Reading the table.**
- `n` is the number of evaluated scenarios. Few-shot conditions are evaluated only where matching examples exist (Table 1).
- `manual` is the unassisted cost on those same scenarios, so compare methods by their *reduction*.
- Results are deterministic: the folds are fixed (`configs/splits/`) and the LLM outputs are cached.